# SageMaker pipeline definition and validation

This sanitized notebook reconstructs the validated three-step experiment workflow. It contains **no original execution outputs, cloud account identifiers, or personal identifiers**. For measured results, consult `results/` and `docs/02-experiments.md`.

Use it from the project root in an AWS-authenticated environment with SageMaker SDK 3.26.0. Definition generation may upload code artifacts to S3 but does not start Processing/Training jobs.

In [ ]:
import os
import json
from pathlib import Path

root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    os.chdir(root.parent)

bucket = os.environ.get("BC_S3_BUCKET")
role = os.environ.get("BC_SAGEMAKER_ROLE_ARN")
region = os.environ.get("AWS_REGION", "ap-northeast-2")
print("Cloud configuration present:", bool(bucket and role))
print("Region:", region)

Before generating an AWS definition, configure your own `BC_S3_BUCKET` and `BC_SAGEMAKER_ROLE_ARN`. Obtain role values privately in your runtime; never commit them to a notebook.

In [ ]:
from src.sagemaker_pipeline import build_pipeline

if not bucket or not role:
    print("No cloud configuration: skipping SageMaker definition")
else:
    pipeline = build_pipeline(bucket=bucket, role=role, region=region)
    definition = json.loads(pipeline.definition())
    print("Pipeline:", pipeline.name)
    print("Steps:", [s["Name"] for s in definition["Steps"]])
    print("Parameters:", [p["Name"] for p in definition["Parameters"]])
    step_eval = next(s for s in definition["Steps"] if s["Name"] == "Evaluate")
    print("PropertyFiles:", step_eval.get("PropertyFiles", []))

## Executions and evaluation outputs

This notebook intentionally **does not** call `pipeline.start()` or `upsert()` to avoid accidental cloud charges. Use `scripts/upsert_pipeline.py --apply` to register the Pipeline; `scripts/launch_experiment.py --execute` starts one explicitly requested run. Use `scripts/inspect_results.py` for the five recorded evaluation JSON files.

In [ ]:
import pandas as pd
from pathlib import Path

results = pd.read_csv(Path("results") / "experiment_comparison_5runs.csv")
display(results)